In [1]:
import sqlite3
import pandas as pd
import numpy as np
import os

# Create artifacts directory if not exists
os.makedirs("artifacts", exist_ok=True)

# Build absolute path to olist.db inside Task-01
base_dir = os.path.dirname(os.path.abspath("__file__"))
db_path = os.path.abspath(os.path.join(base_dir, "..", "Task-01", "olist.db"))

print(f"Connecting to DB at: {db_path}")
conn = sqlite3.connect(db_path)
print("Connected to database successfully!")

Connecting to DB at: c:\Users\sa\Desktop\MLOps-Qafza-2026\Tasks\Task-01\olist.db
Connected to database successfully!


In [2]:
df_orders = pd.read_sql_query("SELECT * FROM olist_orders", conn)
df_customers = pd.read_sql_query("SELECT * FROM olist_customers", conn)

print(f"Orders shape: {df_orders.shape}")
print(f"Customers shape: {df_customers.shape}")

Orders shape: (99441, 8)
Customers shape: (99441, 5)


In [3]:
df_items = pd.read_sql_query("SELECT * FROM olist_order_items", conn)

df_items_agg = df_items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    num_items=('order_item_id', 'count'),
    num_unique_sellers=('seller_id', 'nunique')
).reset_index()

print(f"Aggregated Items shape: {df_items_agg.shape}")

Aggregated Items shape: (98666, 5)


In [4]:
# Aggregate payments to order_id level
df_payments = pd.read_sql_query("SELECT * FROM olist_order_payments", conn)

df_payments_agg = df_payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    max_installments=('payment_installments', 'max'),
    main_payment_type=('payment_type', lambda x: x.mode()[0] if not x.empty else np.nan)
).reset_index()

print(f"Aggregated Payments shape: {df_payments_agg.shape}")

Aggregated Payments shape: (99440, 4)


In [5]:
# Join orders + customers + aggregated items + aggregated payments
ml_df = df_orders.merge(df_customers, on='customer_id', how='left') \
                 .merge(df_items_agg, on='order_id', how='left') \
                 .merge(df_payments_agg, on='order_id', how='left')

print(f"Final ML Table shape: {ml_df.shape}")
print(f"Unique orders: {ml_df['order_id'].nunique()}")

Final ML Table shape: (99441, 19)
Unique orders: 99441


In [6]:
# Save artifact for Notebook 2
artifact_path = os.path.join("artifacts", "ml_table.csv")
ml_df.to_csv(artifact_path, index=False)

print(f"Artifact saved successfully to: {artifact_path}")

Artifact saved successfully to: artifacts\ml_table.csv
